In [ ]:
pip install pandas openpyxl

In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import datetime as dt
import plotly.express as px

import bmll2 as b2
from bmll2 import reference, Security, NormalisedSecurity, SparkHelper, get_market_data, get_market_data_range, VenueMarketError, save_spark_dataframe, load_spark_dataframe, get_market_tables, get_df_engines

pd.options.display.max_columns = None

In [ ]:

def get_data(exchange, ticker, starting_date, ending_date):

    try:
        trades_plus = get_market_data_range([exchange], start_date = starting_date, end_date = ending_date,
                                   table_name = 'trades-plus', df_engine = 'polars', ticker = ticker,
                                   columns = ['Classification', 'MIC', 'Ticker', 'ListingId', 'TradeDate',
                                              'LocalTradeTimestamp', 'ExchangeSequenceNo', 'AggressorSide',
                                              'Price', 'Size', 'BestBidPriceAtPrimary', 'BestAskPriceAtPrimary',
                                              'PreTradeMid1msAtPrimary', 'PostTradeMid1msAtPrimary', 'PostTradeMidAtPrimary'])

    except VenueMarketError:
        return None
    
    trades_plus = trades_plus.filter((pl.col('LocalTradeTimestamp').dt.time() >= pl.time(9, 10)) &
                   (pl.col('LocalTradeTimestamp').dt.time() <= pl.time(16, 50)))
    trades_plus = trades_plus.filter(pl.col('Classification') == 'LIT_CONTINUOUS')
    
    if trades_plus is None or trades_plus.is_empty():
        return None

    columns     = ['MIC', 'Ticker', 'ListingId', 'TradeDate', 'LocalTradeTimestamp', 'ExchangeSequenceNo',
                   'AggressorSide', 'Price', 'Size', 'BestBidPriceAtPrimary', 'BestAskPriceAtPrimary',
                   'PreTradeMid1msAtPrimary', 'PostTradeMidAtPrimary', 'PostTradeMid1msAtPrimary']
    trades_plus = trades_plus.select(columns)
    trades_plus = trades_plus.with_columns((0.5 * (pl.col('BestBidPriceAtPrimary') + pl.col('BestAskPriceAtPrimary'))).alias('Mid-price before'))
    
    trades_plus = trades_plus.with_columns(pl.col('AggressorSide').replace({1: 1, 2: -1, 0: 0}).alias('Trade Sign'))
    trades_plus = trades_plus.filter(pl.col('Trade Sign').is_in([-1, 1]))
    trades_plus = trades_plus.rename({'PreTradeMid1msAtPrimary' : 'Mid-price before (1ms)',
                                      'PostTradeMid1msAtPrimary' : 'Mid-price after (1ms)',
                                      'PostTradeMidAtPrimary' : 'Mid-price after',
                                      'Size' : 'Volume', 'LocalTradeTimestamp' : 'DateTime', 'TradeDate' : 'Date'})
    trades_plus = trades_plus.with_columns(pl.col('Mid-price before').shift(-1).alias('Mid-price after (delayed)'))
    trades_plus = trades_plus.with_columns(pl.col('Mid-price before (1ms)').shift(-1).alias('Mid-price after (1ms)(delayed)'))
    
    columns     = ['MIC', 'Ticker', 'ListingId', 'Date', 'DateTime', 'ExchangeSequenceNo',
                   'Trade Sign', 'Price', 'Volume', 'Mid-price before (1ms)', 'Mid-price after (1ms)', 'Mid-price after (1ms)(delayed)',
                   'Mid-price before', 'Mid-price after', 'Mid-price after (delayed)']
    trades_plus = trades_plus.select(columns)

                                           
    l1 = get_market_data_range('XJSE', start_date = starting_date, end_date = ending_date, ticker = 'GRT',
                                   table_name = 'l1', df_engine = 'polars')
    
    l1 = l1.filter((pl.col('LocalTimestamp').dt.time() >= pl.time(9, 10)) &
                   (pl.col('LocalTimestamp').dt.time() <= pl.time(16, 50)))
    l1 = l1.filter(pl.col('MarketState') == 'CONTINUOUS_TRADING')
    l1 = l1.with_columns(((pl.col('AskPrice1') + pl.col('BidPrice1')) / 2).alias('Mid-price'))
    l1 = l1.rename({'TradeDate' : 'Date'})
    l1 = l1.sort(['Date', 'ExchangeSequenceNo'])
    
    daily_volatility = l1.group_by('Date').agg(((pl.col('Mid-price').max() - pl.col('Mid-price').min()) / pl.col('Mid-price').drop_nulls().first()).alias('Intraday Volatility'))
    daily_volume = trades_plus.group_by('Date').agg(pl.col('Volume').sum().alias('Daily Volume'))

    trades = trades_plus.join(daily_volatility, on = 'Date', how = 'left').join(daily_volume, on = 'Date', how = 'left')

    data  = trades.clone()
    daily = data.group_by(['Date']).agg([pl.col('Daily Volume').first().alias('Daily Volume'),
                                         pl.col('Intraday Volatility').first().alias('Intraday Volatility')]).sort(['Date'])
    
    n     = 20
    daily = daily.with_columns([pl.col('Daily Volume').rolling_mean(n, min_samples = 1).alias(f'{n} AD volume'),
                                pl.col('Intraday Volatility').rolling_mean(n, min_samples = 1).alias(f'{n} AD volatility')])
    trades = trades.join(daily.select(['Date', f'{n} AD volume', f'{n} AD volatility']), on = ['Date'], how = 'left')
    
    trades = trades.sort(['DateTime', 'ExchangeSequenceNo'])
    return trades
    

In [ ]:

b2.get_file('companies_list.csv')

JSE_listed_securities = pl.read_csv('companies_list.csv')
JSE_listed_securities = JSE_listed_securities.with_columns([pl.col('Code').str.strip_chars().str.replace('JSE:', '').alias('Code'),
                                                            pl.col('Market Cap').str.strip_chars().cast(pl.Float64).alias('Market Cap')])
JSE_listed_securities = JSE_listed_securities.rename({'Code': 'Ticker'})
JSE_listed_securities = JSE_listed_securities.sort('Market Cap', descending = True)

top_50 = JSE_listed_securities[0:50, :]
top_100 = JSE_listed_securities[0:100, :]
top_250 = JSE_listed_securities[0:250, :]

top_50.write_csv('top_50.csv')
top_100.write_csv('top_100.csv')
top_250.write_csv('top_250.csv')

b2.put_file('top_50.csv')
b2.put_file('top_100.csv')
b2.put_file('top_250.csv')


In [ ]:

exchange = 'XJSE'
ticker = 'GRT'
starting_date = '2023-01-03'
ending_date = '2023-01-31'


In [ ]:

start_dates = pl.date_range(start = dt.date(2023, 1, 1), end = dt.date(2025, 12, 1),
                            interval = '12mo', closed = 'left', eager = True)
end_dates   = start_dates.dt.offset_by('12mo').dt.offset_by('-1d')


In [ ]:

b2.create_folder('data', ensure_exists = True)
b2.create_folder('data/trades', ensure_exists = True)
b2.create_folder('data/trades/top_250', ensure_exists = True)


In [ ]:
top_250_tickers = top_250['Ticker'].to_list()

In [ ]:

%%time
# A2XX or XJSE
exchange = 'XJSE'
counter = 0
for ticker in top_250_tickers:
    counter += 1
    stock = []
    for i in range(len(start_dates)):
        data = get_data(exchange, ticker, start_dates[i], end_dates[i])
        if data is not None and not data.is_empty():
            stock.append(data)

    if len(stock) == 0:
        continue

    stock_data = pl.concat(stock, how = 'vertical')
    stock_data = stock_data.sort(['DateTime', 'ExchangeSequenceNo'])
    stock_data.write_parquet(f'{ticker}_{exchange}.parquet')
    b2.put_file(f'{ticker}_{exchange}.parquet', 'data/trades/top_250')
    print(f'{counter}: {ticker}')
    